In [11]:
#Importar las librerias
import pandas as pd
import numpy as np
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Detectar dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")

Dispositivo: cuda


In [12]:
# Cargar las tablas principales
patients   = pd.read_csv('./datasets/mimic-iii-clinical-database/PATIENTS.csv')
admissions = pd.read_csv('./datasets/mimic-iii-clinical-database/ADMISSIONS.csv')
icustays   = pd.read_csv('./datasets/mimic-iii-clinical-database/ICUSTAYS.csv')

# Verificar dimensiones de cada tabla
print("PATIENTS:")
print(f"  Filas: {patients.shape[0]}, Columnas: {patients.shape[1]}")
print(f"  Columnas: {patients.columns.tolist()}\n")

print("ADMISSIONS:")
print(f"  Filas: {admissions.shape[0]}, Columnas: {admissions.shape[1]}")
print(f"  Columnas: {admissions.columns.tolist()}\n")

print("ICUSTAYS:")
print(f"  Filas: {icustays.shape[0]}, Columnas: {icustays.shape[1]}")
print(f"  Columnas: {icustays.columns.tolist()}\n")

# Unir las tres tablas por SUBJECT_ID y HADM_ID
df = pd.merge(admissions, patients, on='subject_id', how='left')
df = pd.merge(df, icustays, on=['subject_id', 'hadm_id'], how='left')

print(f"Dataset combinado: {df.shape}")
print(f"\nColumnas del dataset combinado:")
print(df.columns.tolist())

PATIENTS:
  Filas: 100, Columnas: 8
  Columnas: ['row_id', 'subject_id', 'gender', 'dob', 'dod', 'dod_hosp', 'dod_ssn', 'expire_flag']

ADMISSIONS:
  Filas: 129, Columnas: 19
  Columnas: ['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospital_expire_flag', 'has_chartevents_data']

ICUSTAYS:
  Filas: 136, Columnas: 12
  Columnas: ['row_id', 'subject_id', 'hadm_id', 'icustay_id', 'dbsource', 'first_careunit', 'last_careunit', 'first_wardid', 'last_wardid', 'intime', 'outtime', 'los']

Dataset combinado: (136, 36)

Columnas del dataset combinado:
['row_id_x', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospi

In [13]:
# Seleccionar solo las columnas útiles para predecir mortalidad
columnas = [
    'subject_id',          # ID del paciente
    'gender',              # género del paciente
    'admission_type',      # tipo de admisión (EMERGENCY, ELECTIVE, etc.)
    'admission_location',  # de dónde viene el paciente
    'insurance',           # tipo de seguro médico
    'marital_status',      # estado civil
    'expire_flag',# variable objetivo: 0=sobrevivió, 1=falleció
    'los',                 # duración de la estancia en UCI (days)
]

# Filtrar solo columnas que existen en el dataset
columnas_existentes = [c for c in columnas if c in df.columns]
df = df[columnas_existentes].copy()

print(f"Columnas seleccionadas: {df.columns.tolist()}")
print(f"\nShape: {df.shape}")
print(f"\nPrimeras 5 filas:")
print(df.head())

Columnas seleccionadas: ['subject_id', 'gender', 'admission_type', 'admission_location', 'insurance', 'marital_status', 'expire_flag', 'los']

Shape: (136, 8)

Primeras 5 filas:
   subject_id gender admission_type         admission_location insurance  \
0       10006      F      EMERGENCY       EMERGENCY ROOM ADMIT  Medicare   
1       10011      F      EMERGENCY  TRANSFER FROM HOSP/EXTRAM   Private   
2       10013      F      EMERGENCY  TRANSFER FROM HOSP/EXTRAM  Medicare   
3       10017      F      EMERGENCY       EMERGENCY ROOM ADMIT  Medicare   
4       10019      M      EMERGENCY  TRANSFER FROM HOSP/EXTRAM  Medicare   

  marital_status  expire_flag      los  
0      SEPARATED            1   1.6325  
1         SINGLE            1  13.8507  
2            NaN            1   2.6499  
3       DIVORCED            1   2.1436  
4       DIVORCED            1   1.2938  


In [14]:
# Convertir fechas a datetime
admissions['admittime']  = pd.to_datetime(admissions['admittime'])
admissions['dischtime']  = pd.to_datetime(admissions['dischtime'])
admissions['deathtime']  = pd.to_datetime(admissions['deathtime'])

# Calcular duración de la hospitalización en horas
admissions['duracion_horas'] = (
    admissions['dischtime'] - admissions['admittime']
).dt.total_seconds() / 3600

print(f"Duración promedio hospitalización: {admissions['duracion_horas'].mean():.1f} horas")
print(f"Duración mínima: {admissions['duracion_horas'].min():.1f} horas")
print(f"Duración máxima: {admissions['duracion_horas'].max():.1f} horas")



# Unir admissions con df usando una columna común
df = df.merge(admissions[['subject_id', 'duracion_horas']], on='subject_id', how='left')

Duración promedio hospitalización: 224.0 horas
Duración mínima: 0.9 horas
Duración máxima: 2975.6 horas


In [15]:
# Ver cuántos nulos hay por columna
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(1)
resumen_nulos = pd.DataFrame({
    'nulos': nulos,
    'porcentaje': porcentaje
})
resumen_nulos = resumen_nulos[resumen_nulos['nulos'] > 0].sort_values('porcentaje', ascending=False)
print("Columnas con valores nulos:")
print(resumen_nulos)

# Imputar nulos en columna numérica LOS con la mediana
if 'LOS' in df.columns:
    mediana_los = df['LOS'].median()
    df['LOS'] = df['LOS'].fillna(mediana_los)
    print(f"\nMediana LOS usada para imputar: {mediana_los:.2f}")

# Imputar nulos en columnas categóricas con 'Unknown'
cols_categoricas = df.select_dtypes(include='object').columns
for col in cols_categoricas:
    df[col] = df[col].fillna('Unknown')

print(f"\nNulos restantes: {df.isnull().sum().sum()}")

Columnas con valores nulos:
                nulos  porcentaje
marital_status     18         4.7

Nulos restantes: 0


In [16]:
# Identificar tipos de variables
numericas   = df.select_dtypes(include='number').columns.tolist()
categoricas = df.select_dtypes(include='object').columns.tolist()

print(f"Variables numéricas ({len(numericas)}):")
print(numericas)
print(f"\nVariables categóricas ({len(categoricas)}):")
print(categoricas)

# Ver valores únicos de cada categórica
for col in categoricas:
    print(f"\n{col} — {df[col].nunique()} valores únicos:")
    print(df[col].value_counts())

Variables numéricas (4):
['subject_id', 'expire_flag', 'los', 'duracion_horas']

Variables categóricas (5):
['gender', 'admission_type', 'admission_location', 'insurance', 'marital_status']

gender — 2 valores únicos:
gender
M    309
F     73
Name: count, dtype: int64

admission_type — 3 valores únicos:
admission_type
EMERGENCY    370
ELECTIVE      10
URGENT         2
Name: count, dtype: int64

admission_location — 5 valores únicos:
admission_location
EMERGENCY ROOM ADMIT         183
CLINIC REFERRAL/PREMATURE    139
TRANSFER FROM HOSP/EXTRAM     32
TRANSFER FROM SKILLED NUR     15
PHYS REFERRAL/NORMAL DELI     13
Name: count, dtype: int64

insurance — 4 valores únicos:
insurance
Medicare      335
Private        37
Medicaid        9
Government      1
Name: count, dtype: int64

marital_status — 7 valores únicos:
marital_status
MARRIED              283
SINGLE                41
WIDOWED               22
Unknown               18
UNKNOWN (DEFAULT)     11
DIVORCED               6
SEPARATED    

In [17]:
# GENDER solo tiene 2 valores — reemplazar directamente
df['gender'] = df['gender'].map({'M': 0, 'F': 1})
print("gender codificado: M=0, F=1")

# ADMISSION_TYPE — pocos valores, codificación manual
print(f"\nadmission_type valores únicos:")
print(df['admission_type'].value_counts())

tipo_admision = {
    'emergency':        0,
    'elective':         1,
    'urgent':           2,
    'newborn':          3,
    'unknown':          4
}
df['admission_type'] = df['admission_type'].map(tipo_admision).fillna(4)

# INSURANCE, MARITAL_STATUS, ADMISSION_LOCATION
# tienen más valores — usar One Hot Encoding
cols_ohe = ['insurance', 'marital_status', 'admission_location']
cols_ohe_existentes = [c for c in cols_ohe if c in df.columns]

df = pd.get_dummies(df, columns=cols_ohe_existentes, drop_first=True, dtype=int)
print(f"\nShape después de One Hot Encoding: {df.shape}")
print(f"\nColumnas finales:")
print(df.columns.tolist())

gender codificado: M=0, F=1

admission_type valores únicos:
admission_type
EMERGENCY    370
ELECTIVE      10
URGENT         2
Name: count, dtype: int64

Shape después de One Hot Encoding: (382, 19)

Columnas finales:
['subject_id', 'gender', 'admission_type', 'expire_flag', 'los', 'duracion_horas', 'insurance_Medicaid', 'insurance_Medicare', 'insurance_Private', 'marital_status_MARRIED', 'marital_status_SEPARATED', 'marital_status_SINGLE', 'marital_status_UNKNOWN (DEFAULT)', 'marital_status_Unknown', 'marital_status_WIDOWED', 'admission_location_EMERGENCY ROOM ADMIT', 'admission_location_PHYS REFERRAL/NORMAL DELI', 'admission_location_TRANSFER FROM HOSP/EXTRAM', 'admission_location_TRANSFER FROM SKILLED NUR']


In [18]:
# Separar variable objetivo
y = df['expire_flag'].values
X = df.drop(columns=['expire_flag', 'subject_id']).values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Verificar tipos de datos antes de normalizar
print("Tipos de datos en el dataframe:")
print(df.dtypes)

# Ver si quedaron columnas no numéricas
no_numericas = df.select_dtypes(exclude='number').columns.tolist()
print(f"\nColumnas no numéricas que quedaron: {no_numericas}")

# Normalización
def featureNormalize(X):
    mu    = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    return (X - mu) / sigma, mu, sigma

# Convertir booleanos y enteros a float para que NumPy sea feliz
X = X.astype(float)

# Split primero
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Normalizar con datos de train
X_train_norm, mu, sigma = featureNormalize(X_train)
X_test_norm  = (X_test - mu) / sigma

print(f"\nEntrenamiento: {X_train_norm.shape}")
print(f"Prueba:        {X_test_norm.shape}")
print(f"\nMedia X_train_norm (debe ser ~0): {X_train_norm.mean():.6f}")
print(f"Std X_train_norm  (debe ser ~1): {X_train_norm.std():.6f}")

X shape: (382, 17)
y shape: (382,)
Tipos de datos en el dataframe:
subject_id                                        int64
gender                                            int64
admission_type                                  float64
expire_flag                                       int64
los                                             float64
duracion_horas                                  float64
insurance_Medicaid                                int64
insurance_Medicare                                int64
insurance_Private                                 int64
marital_status_MARRIED                            int64
marital_status_SEPARATED                          int64
marital_status_SINGLE                             int64
marital_status_UNKNOWN (DEFAULT)                  int64
marital_status_Unknown                            int64
marital_status_WIDOWED                            int64
admission_location_EMERGENCY ROOM ADMIT           int64
admission_location_PHYS REFERRAL/NORM

In [ ]:
class OpportunityDataset(Dataset):
    """
    Dataset personalizado para el dataset Opportunity.
    Convierte los arrays de NumPy a tensores de PyTorch.
    """
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        # cuántos ejemplos tiene el dataset
        return len(self.X)

    def __getitem__(self, idx):
        # cómo obtener UN ejemplo dado su índice
        return self.X[idx], self.y[idx]


# Crear datasets de train y test
train_dataset = OpportunityDataset(X_train_norm, y_train)
test_dataset  = OpportunityDataset(X_test_norm,  y_test)

# Crear dataloaders
dataloader = {
    'train': DataLoader(train_dataset, batch_size=256, shuffle=True,  pin_memory=True),
    'test':  DataLoader(test_dataset,  batch_size=256, shuffle=False, pin_memory=True)
}

print(f"Ejemplos de entrenamiento: {len(train_dataset):,}")
print(f"Ejemplos de prueba:        {len(test_dataset):,}")
print(f"Batches por época (train): {len(dataloader['train'])}")
print(f"Batches por época (test):  {len(dataloader['test'])}")

# Verificar forma de un batch
X_batch, y_batch = next(iter(dataloader['train']))
print(f"\nForma de un batch — X: {X_batch.shape}, y: {y_batch.shape}")

In [ ]:
class OpportunityNet(nn.Module):
    #Red neuronal multicapa para clasificación de binaria

    def __init__(self, n_inputs=17, n_outputs=2):
        super().__init__()

        self.red = nn.Sequential(
            # Capa 1
            nn.Linear(n_inputs, 100),
            nn.ReLU(),

            # Capa 2
            nn.Linear(100, 100),
            nn.ReLU(),

            # Capa de salida
            nn.Linear(100, n_outputs)
        )

    def forward(self, x):
        return self.red(x)


# Instanciar y mover al dispositivo
model = OpportunityNet(n_inputs=17, n_outputs=2).to(device)

# Ver arquitectura completa
print(model)

# Contar parámetros entrenables
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParámetros entrenables: {total_params:,}")

# Prueba rápida con el batch que obtuvimos antes
with torch.no_grad():
    salida = model(X_batch.to(device))
print(f"\nForma de la salida (logits): {salida.shape}")

OpportunityNet(
  (red): Sequential(
    (0): Linear(in_features=242, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=5, bias=True)
  )
)

Parámetros entrenables: 103,685


NameError: name 'X_batch' is not defined

In [ ]:
def fit(model, dataloader, epochs=30, checkpoint_path='./checkpoint.pt'):
    
    model.to(device)
    
    # Adam adapta el learning rate por peso automáticamente
    # lr=1e-3 es el valor estándar para Adam
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # CrossEntropyLoss incluye Softmax internamente
    # NO aplicar Softmax antes de pasarle los logits
    criterion = nn.CrossEntropyLoss()
    
    historial = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   []
    }
    
    mejor_val_acc = 0.0
    
    for epoch in range(1, epochs + 1):
        
        # ── ENTRENAMIENTO ──────────────────────────────
        model.train()
        train_loss, train_acc = [], []
        
        bar = tqdm(dataloader['train'])
        for X_batch, y_batch in bar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()       # 1. limpiar gradientes
            y_hat = model(X_batch)      # 2. forward
            loss = criterion(y_hat, y_batch)  # 3. calcular pérdida
            loss.backward()             # 4. backward
            optimizer.step()            # 5. actualizar pesos
            
            train_loss.append(loss.item())
            
            preds = torch.argmax(y_hat, axis=1)
            acc   = (preds == y_batch).sum().item() / len(y_batch)
            train_acc.append(acc)
            
            bar.set_description(
                f"Epoch {epoch}/{epochs} [train] "
                f"loss {np.mean(train_loss):.4f} acc {np.mean(train_acc):.4f}"
            )
        
        # ── EVALUACIÓN ─────────────────────────────────
        model.eval()
        val_loss, val_acc = [], []
        
        with torch.no_grad():
            bar = tqdm(dataloader['test'])
            for X_batch, y_batch in bar:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                y_hat = model(X_batch)
                loss  = criterion(y_hat, y_batch)
                
                val_loss.append(loss.item())
                
                preds = torch.argmax(y_hat, axis=1)
                acc   = (preds == y_batch).sum().item() / len(y_batch)
                val_acc.append(acc)
                
                bar.set_description(
                    f"Epoch {epoch}/{epochs} [val]  "
                    f"val_loss {np.mean(val_loss):.4f} val_acc {np.mean(val_acc):.4f}"
                )
        
        # Promediar métricas de la época
        epoch_train_loss = np.mean(train_loss)
        epoch_train_acc  = np.mean(train_acc)
        epoch_val_loss   = np.mean(val_loss)
        epoch_val_acc    = np.mean(val_acc)
        
        historial['train_loss'].append(epoch_train_loss)
        historial['train_acc'].append(epoch_train_acc)
        historial['val_loss'].append(epoch_val_loss)
        historial['val_acc'].append(epoch_val_acc)
        
        # ── CHECKPOINT ─────────────────────────────────
        # Guardar solo si mejoró val_acc respecto al mejor hasta ahora
        if epoch_val_acc > mejor_val_acc:
            mejor_val_acc = epoch_val_acc
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc':              epoch_val_acc,
                'val_loss':             epoch_val_loss,
            }, checkpoint_path)
            print(f"  ✓ Checkpoint guardado — epoch {epoch} val_acc {epoch_val_acc:.4f}")
        
        print(
            f"Epoch {epoch}/{epochs} | "
            f"loss {epoch_train_loss:.4f} | acc {epoch_train_acc:.4f} | "
            f"val_loss {epoch_val_loss:.4f} | val_acc {epoch_val_acc:.4f}"
        )
    
    print(f"\nMejor val_acc lograda: {mejor_val_acc:.4f}")
    return historial

In [ ]:
# Instanciar modelo fresco
model = OpportunityNet(n_inputs=242, n_outputs=5).to(device)

# Entrenar — el mejor checkpoint se guarda automáticamente en ./checkpoint.pt
historial = fit(
    model,
    dataloader,
    epochs=30,
    checkpoint_path='./checkpoint.pt'
)

In [ ]:
# Cargar el mejor modelo guardado durante el entrenamiento
checkpoint = torch.load('checkpoint.pt', map_location=device, weights_only=False)

# Restaurar pesos en el modelo
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Checkpoint cargado:")
print(f"  Época:    {checkpoint['epoch']}")
print(f"  val_acc:  {checkpoint['val_acc']:.4f}")
print(f"  val_loss: {checkpoint['val_loss']:.4f}")

# Evaluación completa sobre test
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in dataloader['test']:
        X_batch = X_batch.to(device)
        y_hat   = model(X_batch)
        preds   = torch.argmax(y_hat, axis=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc_pytorch = accuracy_score(all_labels, all_preds) * 100
print(f"\nAccuracy final (test): {acc_pytorch:.2f}%")

# Reporte detallado por clase
nombres_clases = ['Relaxing', 'Coffee time', 'Early morning', 'Cleanup', 'Sandwich time']
print("\nReporte de clasificación:")
print(classification_report(all_labels, all_preds, target_names=nombres_clases))

In [ ]:


# ── Gráfica 1: Curvas de entrenamiento ──────────────────────
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(historial['train_loss']) + 1)

ax1.plot(epochs_range, historial['train_loss'], label='Train loss', color='steelblue')
ax1.plot(epochs_range, historial['val_loss'],   label='Val loss',   color='coral')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.set_title('Pérdida por época')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, [a*100 for a in historial['train_acc']], label='Train acc', color='steelblue')
ax2.plot(epochs_range, [a*100 for a in historial['val_acc']],   label='Val acc',   color='coral')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy por época')
ax2.legend()
ax2.grid(True)

pyplot.tight_layout()
pyplot.show()

# ── Gráfica 2: Comparación con modelo anterior ──────────────
# Reemplaza estos valores con los de tu laboratorio anterior
acc_onevsall_train = 0.0   # <-- pega tu precisión de entrenamiento
acc_onevsall_test  = 0.0   # <-- pega tu precisión de prueba

acc_pytorch_train = np.mean(historial['train_acc']) * 100

modelos   = ['OneVsAll\n(Lab anterior)', 'Red Neuronal\n(PyTorch)']
acc_train = [acc_onevsall_train, acc_pytorch_train]
acc_test  = [acc_onevsall_test,  acc_pytorch]

x     = np.arange(len(modelos))
width = 0.35

fig, ax = pyplot.subplots(figsize=(8, 6))
barras_train = ax.bar(x - width/2, acc_train, width, label='Train', color='steelblue')
barras_test  = ax.bar(x + width/2, acc_test,  width, label='Test',  color='coral')

for barra in list(barras_train) + list(barras_test):
    altura = barra.get_height()
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        altura + 0.5,
        f'{altura:.2f}%',
        ha='center', va='bottom', fontsize=10
    )

ax.set_ylabel('Accuracy (%)')
ax.set_title('Comparación: OneVsAll vs Red Neuronal PyTorch')
ax.set_xticks(x)
ax.set_xticklabels(modelos)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis='y')
pyplot.tight_layout()
pyplot.show()